# 03. Data cleaning and standardization

This notebook:
1. Loads raw CSVs from `data/`.
2. Applies **country-name normalization** so joins work across tables.
3. Runs **validation checks** (row counts, nulls, numeric ranges, duplicates).
4. Exports cleaned DataFrames to **`data/processed/`** with schema-friendly column names.

## Paths and imports

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def csv_path(*parts):
    return DATA_DIR.joinpath(*parts)

## Country-name normalization

Mapping from names as they appear in source CSVs to the standard names used for joins. Based on the original (continent/country alignment and special characters).

In [ ]:
COUNTRY_RENAME = {
    "Cape Verde": "Cabo Verde",
    "Curacao": "Curaçao",
    "Czech Republic": "Czech Republic (Czechia)",
    "Faroe Islands": "Faeroe Islands",
    "Macau": "Macao",
    "Republic of the Congo": "Congo",
    "Reunion": "Réunion",
    "Saint Kitts and Nevis": "Saint Kitts & Nevis",
    "Saint Pierre and Miquelon": "Saint Pierre & Miquelon",
    "Saint Vincent and the Grenadines": "St. Vincent & Grenadines",
    "Palestine": "State of Palestine",
    "Sao Tome and Principe": "Sao Tome & Principe",
    "Turks and Caicos Islands": "Turks and Caicos",
    "United States Virgin Islands": "U.S. Virgin Islands",
    "Wallis and Futuna": "Wallis & Futuna",
    "Vatican City": "Holy See",
    "Guernsey": "Channel Islands",
    "Jersey": "Channel Islands",
    "Ivory Coast": "Côte d'Ivoire",
}

# Ivory Coast: any value containing "Côte" is standardized to "Côte dIvoire".
def normalize_country_column(series):
    s = series.replace(COUNTRY_RENAME)
    mask = s.astype(str).str.contains("Côte", na=False)
    s = s.where(~mask, "Côte dIvoire")
    return s

## Load and clean each table

Load raw CSVs, apply country normalization to the country column ("Country" or "country"), then run basic validation.

In [ ]:
# Continents
continents = pd.read_csv(csv_path("continents.csv"))
continents["country"] = normalize_country_column(continents["country"])
# After mapping, Guernsey and Jersey both become "Channel Islands"; keep one row.
continents = continents.drop_duplicates(subset=["country"], keep="first")
# Add rows that may be missing in raw data but appear in other tables.
extra = [
    {"country": "Caribbean Netherlands", "continent": "Europe", "subregion": "Caribbean"},
    {"country": "Saint Helena", "continent": "Europe", "subregion": "Western Africa, Sub-Saharan Africa"},
]
for row in extra:
    if row["country"] not in continents["country"].values:
        continents = pd.concat([continents, pd.DataFrame([row])], ignore_index=True)

print("Continents:", continents.shape)
continents.head()

In [ ]:
# Economy
economy = pd.read_csv(csv_path("economy.csv"))
economy["Country"] = normalize_country_column(economy["Country"])
print("Economy:", economy.shape)
economy.head()

In [ ]:
# Population
population = pd.read_csv(csv_path("population.csv"))
population["Country"] = normalize_country_column(population["Country"])
print("Population:", population.shape)
population.head()

In [ ]:
# Education
education = pd.read_csv(csv_path("education.csv"))
education["Country"] = normalize_country_column(education["Country"])
print("Education:", education.shape)
education.head()

In [ ]:
# Education quality (PISA)
quality_education = pd.read_csv(csv_path("quality_education.csv"))
quality_education["Country"] = normalize_country_column(quality_education["Country"])
print("Education quality:", quality_education.shape)
quality_education.head()

## Validation checks

Run assertions and report so we know the cleaned data is consistent before export and DB load.

In [ ]:
def validate_no_duplicate_countries(df, name, country_col="Country"):
    if country_col not in df.columns:
        country_col = "country"
    dup = df[df.duplicated(subset=[country_col], keep=False)]
    if len(dup) > 0:
        print(f"[WARN] {name}: duplicate {country_col}", dup[country_col].tolist())
    else:
        print(f"[OK] {name}: no duplicate {country_col}")

validate_no_duplicate_countries(continents, "continents", "country")
validate_no_duplicate_countries(economy, "economy")
validate_no_duplicate_countries(population, "population")
validate_no_duplicate_countries(education, "education")
validate_no_duplicate_countries(quality_education, "education_quality")

In [ ]:
# Numeric ranges: enrollment and completion are 0–100; urban_pop can be 0–1.
pct_cols_0_100 = [
    "Avg primary enrollment ratio (%)", "Avg primary completion rate (%)",
    "Avg secondary enrollment ratio (%)", "Avg secondary completion rate (%)",
    "Avg terciary enrollment ratio (%)", "Avg terciary completion rate (%)",
]
for col in pct_cols_0_100:
    if col in education.columns:
        valid = education[col].dropna()
        out = valid[(valid < 0) | (valid > 100)]
        if len(out) > 0:
            print(f"[WARN] education {col}: values outside 0–100", out.tolist())
print("[OK] education: percentage columns in 0–100 or null")

In [ ]:
# Referential sanity: every country in economy/population/education/quality should exist in continents.
continent_countries = set(continents["country"])
for name, df, col in [("economy", economy, "Country"), ("population", population, "Country"),
                      ("education", education, "Country"), ("quality_education", quality_education, "Country")]:
    missing = set(df[col].dropna()) - continent_countries
    if missing:
        print(f"[WARN] {name}: countries not in continents: {sorted(missing)[:10]}{'...' if len(missing) > 10 else ''}")
    else:
        print(f"[OK] {name}: all countries present in continents")

## Export to `data/processed/`

Rename columns to match the SQLite schema so the load step can map them directly. Use `country` as the join key (same name in all exported files).

In [ ]:
# Continents: already has country, continent, subregion
continents.to_csv(PROCESSED_DIR / "continents.csv", index=False)
print("Exported:", PROCESSED_DIR / "continents.csv")

In [ ]:
# Economy: Country -> country, then schema names
economy_export = economy.rename(columns={
    "Country": "country",
    "Population (2020)": "population_2020",
    "GDP per capita 2020 (USD)": "gdp_per_capita_2020_usd",
    "Avg Gov exp education % of GDP (last 20 years)": "avg_gov_exp_edu_gdp_pct_20y",
})
economy_export.to_csv(PROCESSED_DIR / "economy.csv", index=False)
print("Exported:", PROCESSED_DIR / "economy.csv")

In [ ]:
# Population
population_export = population.rename(columns={
    "Country": "country",
    "Population (2020)": "population_2020",
    "Urban Pop %": "urban_pop_pct",
    "GDP per capita 2020 (USD)": "gdp_per_capita_2020_usd",
})
population_export.to_csv(PROCESSED_DIR / "population.csv", index=False)
print("Exported:", PROCESSED_DIR / "population.csv")

In [ ]:
# Education
education_export = education.rename(columns={
    "Country": "country",
    "Avg primary enrollment ratio (%)": "avg_primary_enrollment_pct",
    "Avg primary completion rate (%)": "avg_primary_completion_pct",
    "Avg secondary enrollment ratio (%)": "avg_secondary_enrollment_pct",
    "Avg secondary completion rate (%)": "avg_secondary_completion_pct",
    "Avg terciary enrollment ratio (%)": "avg_tertiary_enrollment_pct",
    "Avg terciary completion rate (%)": "avg_tertiary_completion_pct",
})
education_export.to_csv(PROCESSED_DIR / "education.csv", index=False)
print("Exported:", PROCESSED_DIR / "education.csv")

In [ ]:
# Education quality
quality_export = quality_education.rename(columns={
    "Country": "country",
    "Avg PISA performance_reading": "avg_pisa_reading",
    "Avg PISA performance_mathematics": "avg_pisa_mathematics",
    "Avg PISA performance_science": "avg_pisa_science",
})
quality_export.to_csv(PROCESSED_DIR / "education_quality.csv", index=False)
print("Exported:", PROCESSED_DIR / "education_quality.csv")

## Summary

Cleaned CSVs are in `data/processed/`. Use them in the next step to load the SQLite database (continents first to get `country_id`, then the rest).